In [ ]:
import pandas as pd

file_path = "first_25000_rows.csv"
df = pd.read_csv(file_path)

df.head()

,ts_recv,ts_event,rtype,publisher_id,instrument_id,action,side,depth,price,size,...,ask_sz_08,bid_ct_08,ask_ct_08,bid_px_09,ask_px_09,bid_sz_09,ask_sz_09,bid_ct_09,ask_ct_09,symbol
0,2024-10-21T11:54:29.221230963Z,2024-10-21T11:54:29.221064336Z,10,2,38,C,B,1,233.62,2,...,155,1,7,233.25,234.13,55,400,2,1,AAPL
1,2024-10-21T11:54:29.223936626Z,2024-10-21T11:54:29.223769812Z,10,2,38,A,B,0,233.67,2,...,155,1,7,233.25,234.13,55,400,2,1,AAPL
2,2024-10-21T11:54:29.225196809Z,2024-10-21T11:54:29.225030400Z,10,2,38,A,B,0,233.67,3,...,155,1,7,233.25,234.13,55,400,2,1,AAPL
3,2024-10-21T11:54:29.712600612Z,2024-10-21T11:54:29.712434212Z,10,2,38,A,B,2,233.52,200,...,155,1,7,233.25,234.13,55,400,2,1,AAPL
4,2024-10-21T11:54:29.764839221Z,2024-10-21T11:54:29.764673165Z,10,2,38,C,B,2,233.52,200,...,155,1,7,233.25,234.13,55,400,2,1,AAPL


In [ ]:
# Filter for depth level 0 (best bid/ask)
df_best = df[df['depth'] == 0].copy()

df_best['side_sign'] = df_best['side'].map({'B': 1, 'A': -1})

action_sign = {'A': 1, 'C': -1, 'E': -1}  # Add, Cancel, Execute
df_best['action_sign'] = df_best['action'].map(action_sign)

df_best['ofi_contrib'] = df_best['side_sign'] * df_best['action_sign'] * df_best['size']

df_best.set_index('ts_event', inplace=True)
ofi_best_level = df_best['ofi_contrib'].resample('1S').sum().rename('OFI_BestLevel')

ofi_best_level_df = ofi_best_level.reset_index()

print(ofi_best_level_df.head())

                   ts_event  OFI_BestLevel
0 2024-10-21 11:54:29+00:00            5.0
1 2024-10-21 11:54:30+00:00            0.0
2 2024-10-21 11:54:31+00:00            0.0
3 2024-10-21 11:54:32+00:00            0.0
4 2024-10-21 11:54:33+00:00            0.0


In [ ]:
df['ts_event'] = pd.to_datetime(df['ts_event'])

depth_levels = list(range(5))
df_multi = df[df['depth'].isin(depth_levels)].copy()

df_multi['side_sign'] = df_multi['side'].map({'B': 1, 'A': -1})
df_multi['action_sign'] = df_multi['action'].map({'A': 1, 'C': -1, 'E': -1})
df_multi['ofi_contrib'] = df_multi['side_sign'] * df_multi['action_sign'] * df_multi['size']

df_multi.set_index('ts_event', inplace=True)
multi_level_ofi = (
    df_multi.groupby([pd.Grouper(freq='1S'), 'depth'])['ofi_contrib']
    .sum()
    .unstack(fill_value=0)
)

print(multi_level_ofi.head())


depth                          0      1    2     3     4
ts_event                                                
2024-10-21 11:54:29+00:00    5.0  398.0  0.0   0.0   0.0
2024-10-21 11:54:37+00:00 -200.0    0.0  0.0   0.0   0.0
2024-10-21 11:54:39+00:00    0.0    0.0  0.0   0.0   0.0
2024-10-21 11:54:41+00:00    0.0    0.0  0.0   0.0   0.0
2024-10-21 11:54:50+00:00    0.0    0.0  0.0 -29.0  29.0


In [23]:
from sklearn.decomposition import PCA

multi_level_ofi_clean = multi_level_ofi.dropna()

pca = PCA(n_components=1)
integrated_ofi_values = pca.fit_transform(multi_level_ofi_clean)

# Normalize by L1 norm
w1 = pca.components_[0]
integrated_ofi = integrated_ofi_values.flatten() / abs(w1).sum()

integrated_ofi_df = pd.DataFrame({
    'timestamp': multi_level_ofi_clean.index,
    'Integrated_OFI': integrated_ofi
})
print(integrated_ofi_df.head())


                  timestamp  Integrated_OFI
0 2024-10-21 11:54:29+00:00     -141.382730
1 2024-10-21 11:54:37+00:00      -31.292642
2 2024-10-21 11:54:39+00:00      -16.964027
3 2024-10-21 11:54:41+00:00      -16.964027
4 2024-10-21 11:54:50+00:00      -12.532658


In [24]:
from sklearn.linear_model import LassoCV

df_best = df[df['depth'] == 0].copy()
df_best['side_sign'] = df_best['side'].map({'B': 1, 'A': -1})
df_best['action_sign'] = df_best['action'].map({'A': 1, 'C': -1, 'E': -1})
df_best['ofi_contrib'] = df_best['side_sign'] * df_best['action_sign'] * df_best['size']

df_best.set_index('ts_event', inplace=True)
ofi_all = df_best.groupby([pd.Grouper(freq='1S'), 'symbol'])['ofi_contrib'].sum().unstack().fillna(0)

df['mid'] = (df['bid_px_00'] + df['ask_px_00']) / 2

mid_series = (
    df.set_index('ts_event')
      .groupby('symbol')['mid']
      .resample('1S')
      .last()
      .reset_index()
      .pivot(index='ts_event', columns='symbol', values='mid')
)

returns = mid_series.pct_change().fillna(0)


aligned_idx = ofi_all.index.intersection(returns.index)
ofi_all = ofi_all.loc[aligned_idx]
returns = returns.loc[aligned_idx]

y = returns['AAPL']
X = ofi_all

lasso = LassoCV(cv=5).fit(X, y)

cross_asset_beta = pd.Series(lasso.coef_, index=X.columns)
print(cross_asset_beta.sort_values(ascending=False))


symbol
AAPL    1.874065e-07
dtype: float64
